# Treino local com Llama + Unsloth no Google Colab

Este notebook treina um adapter LoRA/QLoRA para o fluxo de detecção de violência doméstica usando `data/violence_detection_finetuning.jsonl`.

Antes de rodar: em `Runtime > Change runtime type`, selecione GPU. Uma T4 costuma funcionar para o modelo 8B em 4 bits, com batch pequeno.


## 1. Instalar dependências

A instalação pode levar alguns minutos e talvez o Colab peça para reiniciar o runtime. Se reiniciar, rode as células novamente a partir daqui.


In [ ]:
%%capture
!pip install -U unsloth
!pip install -U --no-deps trl peft accelerate bitsandbytes datasets transformers


## 2. Conferir GPU


In [ ]:
!nvidia-smi


Wed May 13 22:48:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Montar Google Drive e definir caminhos

O notebook usa o Drive por padrão para ler o dataset e salvar o adapter/tokenizer treinado.

Caminho esperado no Drive:

```text
MyDrive/techchallenge3/data/violence_detection_finetuning.jsonl
```

Se você fizer upload manual no Colab como `/content/violence_detection_finetuning.jsonl`, a célula abaixo copia esse arquivo para o Drive automaticamente.


In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
PROJECT_DIR = Path('/content/drive/MyDrive/techchallenge3')
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'llama-violence-detection-lora'
DATASET_PATH = DATA_DIR / 'violence_detection_finetuning.jsonl'
LOCAL_UPLOAD_PATH = Path('/content/violence_detection_finetuning.jsonl')

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.exists() and LOCAL_UPLOAD_PATH.exists():
    shutil.copy2(LOCAL_UPLOAD_PATH, DATASET_PATH)
    print(f'Dataset copiado para o Drive: {DATASET_PATH}')

assert DATASET_PATH.exists(), (
    'Dataset não encontrado. Faça upload de violence_detection_finetuning.jsonl para '
    '/content ou salve em MyDrive/techchallenge3/data/'
)

print('Dataset:', DATASET_PATH)
print('Saída do modelo:', OUTPUT_DIR)


Dataset: /content/drive/MyDrive/techchallenge3/data/violence_detection_finetuning.jsonl
Saída do modelo: /content/drive/MyDrive/techchallenge3/outputs/llama-violence-detection-lora


## 4. Carregar e validar o JSONL


In [ ]:
import json
from statistics import mean

def load_chat_jsonl(path):
    examples = []
    with open(path, encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            payload = json.loads(line)
            messages = payload.get('messages')
            if not isinstance(messages, list) or len(messages) < 3:
                raise ValueError(f'Linha {line_number} sem messages válidas')
            for message in messages:
                if message.get('role') not in {'system', 'user', 'assistant'}:
                    raise ValueError(f'Linha {line_number} com role inválido: {message}')
                if not message.get('content'):
                    raise ValueError(f'Linha {line_number} com content vazio: {message}')
            examples.append(payload)
    if not examples:
        raise ValueError('Dataset vazio')
    return examples




In [ ]:
examples = load_chat_jsonl(DATASET_PATH)


In [ ]:
len(examples), examples[0]['messages'][1]

(60,
 {'role': 'user',
  'content': 'Meu marido fica com meu celular e não deixa eu falar com minha família. Ele diz que é para o meu bem.'})

## 5. Carregar Llama com Unsloth

O modelo abaixo é carregado em 4 bits para reduzir VRAM. Você pode trocar por outro modelo compatível do Unsloth.


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

[transformers] Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


## 6. Formatar com chat template e tokenizar preview

Aqui o dataset passa pelo tokenizer. O objetivo é medir tamanho em tokens e verificar se haverá truncamento.


In [ ]:
def render_messages(messages, tokenizer):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

def format_dataset_into_model_input(examples, tokenizer):
    return [
        {
            'text': render_messages(example['messages'], tokenizer),
            'metadata': example.get('metadata', {}),
        }
        for example in examples
    ]




In [ ]:
formatted_examples = format_dataset_into_model_input(examples, tokenizer)

token_lengths = []
truncated = 0
for example in formatted_examples:
    tokens = tokenizer(
        example['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    length = len(tokens['input_ids'])
    token_lengths.append(length)
    if length >= MAX_SEQ_LENGTH:
        truncated += 1

{
    'examples': len(formatted_examples),
    'min_tokens': min(token_lengths),
    'avg_tokens': round(mean(token_lengths), 2),
    'max_tokens': max(token_lengths),
    'truncated_examples': truncated,
}

{'examples': 60,
 'min_tokens': 416,
 'avg_tokens': 436.02,
 'max_tokens': 477,
 'truncated_examples': 0}

In [ ]:
formatted_examples

## 7. Preparar LoRA


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


[transformers] Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 8. Treinar

Para demonstração, `max_steps=60`. Para um treino mais completo, aumente esse valor e monitore perda/overfitting.

A saída do treinamento também usa `OUTPUT_DIR`, que já aponta para o Google Drive.


In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
import inspect

train_dataset = Dataset.from_list(formatted_examples)

sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    'output_dir': str(OUTPUT_DIR),
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 4,
    'warmup_steps': 5,
    'max_steps': 60,
    'learning_rate': 2e-4,
    'fp16': True,
    'logging_steps': 1,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 3407,
    'report_to': 'none',
    'dataset_text_field': 'text',
    'packing': False,
    'padding_free': False,
}

if 'max_length' in sft_params:
    sft_kwargs['max_length'] = MAX_SEQ_LENGTH
elif 'max_seq_length' in sft_params:
    sft_kwargs['max_seq_length'] = MAX_SEQ_LENGTH

sft_kwargs = {key: value for key, value in sft_kwargs.items() if key in sft_params}
sft_config = SFTConfig(**sft_kwargs)

trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    'model': model,
    'train_dataset': train_dataset,
    'args': sft_config,
}

if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_params:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 8 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.369619
2,2.407911
3,2.345687
4,2.177899
5,1.950946
6,1.746169
7,1.519239
8,1.314232
9,0.989804
10,0.855635


[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/techchallenge3/outputs/llama-violence-detection-lora/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.4929360404610634, metrics={'train_runtime': 647.1114, 'train_samples_per_second': 0.742, 'train_steps_per_second': 0.093, 'total_flos': 9074897111384064.0, 'train_loss': 0.4929360404610634})

## 9. Salvar adapter e tokenizer no Google Drive

Esta célula salva explicitamente o adapter LoRA e o tokenizer em `MyDrive/techchallenge3/outputs/llama-violence-detection-lora`.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

saved_files = sorted(path.name for path in OUTPUT_DIR.iterdir())
print('Modelo/tokenizer salvos em:', OUTPUT_DIR)
saved_files


[transformers] Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/techchallenge3/outputs/llama-violence-detection-lora/tokenizer_config.json.


Modelo/tokenizer salvos em: /content/drive/MyDrive/techchallenge3/outputs/llama-violence-detection-lora


['README.md',
 'adapter_config.json',
 'adapter_model.safetensors',
 'chat_template.jinja',
 'checkpoint-60',
 'tokenizer.json',
 'tokenizer_config.json']

## 10. Testar inferência


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        'role': 'system',
        'content': 'Você é um assistente de apoio à triagem em saúde e segurança da mulher. Responda somente em JSON válido com classificação de risco, sinais de alerta, próxima etapa e resposta acolhedora.',
    },
    {
        'role': 'user',
        'content': 'Não aconteceu nada demais, somente um empurrão.',
    },
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Você é um assistente de apoio à triagem em saúde e segurança da mulher. Responda somente em JSON válido com classificação de risco, sinais de alerta, próxima etapa e resposta acolhedora.user

Não aconteceu nada demais, somente um empurrão.assistant

{"classificacao_risco": "moderado", "sinais_alerta": ["agressão", "controle", "risco recorrente"], "proxima_etapa_fluxo": "entrevista_privada", "resposta_acolhedora": "O empurrão é um tipo de agressão que pode causar medo e sofrimento. Se isso voltar a acontecer, procure atendimento médico e psicossocial. Aqui, você é acolhida e sua segurança é priorizada."}


## 11. Compactar adapter para download opcional

Como o modelo já foi salvo no Drive, esta etapa é opcional. Ela cria um `.zip` no próprio Drive para facilitar download/backup.


In [ ]:
ZIP_PATH = OUTPUT_DIR.parent / 'llama-violence-detection-lora.zip'
!cd {OUTPUT_DIR.parent} && zip -r {ZIP_PATH.name} {OUTPUT_DIR.name}
print('ZIP salvo em:', ZIP_PATH)


  adding: llama-violence-detection-lora/ (stored 0%)
  adding: llama-violence-detection-lora/README.md (deflated 46%)
  adding: llama-violence-detection-lora/checkpoint-60/ (stored 0%)
  adding: llama-violence-detection-lora/checkpoint-60/README.md (deflated 65%)
  adding: llama-violence-detection-lora/checkpoint-60/adapter_model.safetensors (deflated 8%)
  adding: llama-violence-detection-lora/checkpoint-60/adapter_config.json (deflated 58%)
  adding: llama-violence-detection-lora/checkpoint-60/chat_template.jinja (deflated 72%)
  adding: llama-violence-detection-lora/checkpoint-60/tokenizer_config.json (deflated 96%)
  adding: llama-violence-detection-lora/checkpoint-60/tokenizer.json (deflated 85%)
  adding: llama-violence-detection-lora/checkpoint-60/training_args.bin (deflated 53%)
  adding: llama-violence-detection-lora/checkpoint-60/optimizer.pt (deflated 11%)
  adding: llama-violence-detection-lora/checkpoint-60/scheduler.pt (deflated 62%)
  adding: llama-violence-detection-lor